## Setup Source File and Target Directory

In [ ]:
source = '~/Downloads/setyl/assets-2025-12-10.csv'
converted_dir = 'converted/'
converted_file_name = 'converted_'+source.split('/')[-1]

## Setup Pandas

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(source)

## Merge Column

In [ ]:
def merge_with_enter(col1, col2, removeColumns=False):
    raw_df = df[col1].fillna('').astype(str).str.cat(
        df[col2].fillna('').astype(str),
        '\n\n',
        na_rep = ''
    )
    if removeColumns:
        df.drop(columns=[col1, col2], inplace=True)
    return raw_df.str.strip()

df['Asset Notes'] = merge_with_enter('Notes', 'Specifics', True)


In [ ]:
def merge_when_empty(main_col, fallback_col, removeColumns=False):
    result_df = df[main_col].fillna(df[fallback_col])
    if removeColumns:
        df.drop(columns=[main_col, fallback_col], inplace=True)
    return result_df

df['Order Number'] = merge_when_empty('Purchase Order No.', 'Invoice Number', True)
df['Operating System'] = merge_when_empty('Operating System Type', 'Operating System', True)
df['Location'] = merge_when_empty('Location', 'Location Type', True)

## Normalize Number Column to Integer instead Float

In [ ]:
def normalize_number(col):
    raw = np.floor(df[col]).astype('Int64')
    clean = raw.apply(lambda x: '' if pd.isna(x) or x == 0 else str(x).split('.')[0])
    return clean

df['Storage Size (GB)'] = normalize_number('Storage Size (GB)')
df['RAM'] = normalize_number('RAM')
df['Screen Size (inches)'] = normalize_number('Screen Size (inches)')
df['Price'] = normalize_number('Price')

## Field Modification

In [ ]:
df['BYOD'] = (df['Ownership'] == 'Company Owned')           # Change cell to True/False if Ownership = Company Owned

## Rename Column

Matching Setyl's with Snipe-it columns

In [ ]:
df.rename(columns={
    'Type': 'Category',
    'Price': 'Purchase Cost',
    'Assignee': 'Checked Out to: Username',
    'Purchased on': 'Purchase Date',
    'End of Useful Life': 'EOL Date',
    'Asset ID': 'Asset Tag',
    # 'Location Type': 'Location'
},
inplace=True)

## Cleanup unused field

In [ ]:
df.drop(columns=[
    'Category (Legacry)',
    'Criticality',
    'External Link',
    'Groups',
    'IP Address',
    'Keyboard Layout',
    'Lease Expiry Date',
    'Leasing Agreement Number',
    'Legacy Asset ID',
    'Legal Entity',
    'Maximum Operating System',
    'MDM Sources',
    'Model Number',
    'Motherboard',
    'Phone Extension',
    'Purchased Currency',
    'Screen Resolution',
    'Sublocation',
    'URL',
    'Useful Life',              # Age in years, replaced with End-of-life date
    'Department',               # User's Department
    'Ownership',                # Replaced with BYOD true/false if company owned
],inplace=True)


## Export to converted CSV

In [ ]:

df.to_csv(converted_dir + '/' + converted_file_name, index=False)